In [0]:
%run ./Common_utils

In [0]:
%run ./api_config

In [0]:
%run ./issues_config

In [0]:
JIRA_USER = "<email>"
JIRA_API_TOKEN = "<token>"
auth = (JIRA_USER, JIRA_API_TOKEN)
headers = {"Accept": "application/json"}

In [0]:
TARGET_DATABASE = "Data_ingestion2"
BRONZE_TABLE = "issues_bronze2"
FLAT_TABLE = "issues_flat2"
RUN_LOG_TABLE = "pipeline_run_log2"

Testing of API


In [0]:
url = build_url(base, version ,endpoint["projects"])
api = api_call(url , auth, headers, params = None)


 Testing of fetch_paginated_data

In [0]:
ans = fetch_paginated_data(
    url=url,auth=auth,
    headers=headers
    ,params={},data_key="values"
      
)

In [0]:
project_url = build_url(base,version,endpoint["projects"])  # Call build_url from Common_utils

projects = fetch_paginated_data(                # Call project API and returns list of all projects in JSON form
    url= project_url,
    auth=auth,
    headers=headers,
    params={},
    data_key="values",
    batch_size=50

)

project_keys =[p["key"] for p in projects if p.get("key")]  # Extracts the project key ("key") from each dictionary of proj from above returned list.

In [0]:
project_keys

In [0]:
issues_url = build_url(base,version,endpoint["issues"])  # Call build_url from Common_utils

all_issues_raw =[]

for key in project_keys:

    jql =f'project = "{key}"'
    #jql = 'project in("MDP","MYK","MYJP")'

    issues = fetch_paginated_data(
         url= issues_url,
         auth=auth,
         headers=headers,
         params={
        "jql" : jql,
         "fields" : "summary,description,issuetype,status,assignee,reporter,priority,parent,customfield_10020,customfield_10016"
         },
         data_key="issues"
    )

    parsed = apply_parser(issues, parse_issue) # call the func "apply_parser" from NB common Utils and pass the func "parse_issue" from NB issues_config

    all_issues_raw.extend(parsed) 
             


In [0]:
all_issues_raw

In [0]:
df_raw = create_df(spark, all_issues_raw) # Call "create_df" from common_utils

df_bronze = (
    df_raw
    .withColumn("parsed", F.from_json("raw_json", jira_schema))
)

df_bronze.select(F.col("parsed.*"))

df_flat = df_bronze.select(
    F.col("parsed.id").alias("ID"),
    F.col("parsed.key").alias("key"),
    F.col("parsed.fields.*")
)

In [0]:
display(df_flat)

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {TARGET_DATABASE}")
write_table(df_flat, f"{TARGET_DATABASE}.{FLAT_TABLE}")

In [0]:
log_pipeline_run(
    spark,
    f"{TARGET_DATABASE}.{RUN_LOG_TABLE}",
    pipeline_name="issues",
    status="SUCCESS",
    records_processed=len(all_issues_raw),
    projects_processed=len(project_keys)
)